# ODI to Databricks Migration: `TRG_DEP_INSERT.txt`

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook performs a full load insert into the `TRG_DEP` target table from the `DEPARTMENTS` source table.

In [ ]:
%sql
-- SCEN_TASK_NO in {10}, {20}, {30}: Initialization tasks (no SQL statements)

-- Create target table if it does not exist (inferred from HR.DEPARTMENTS structure)
CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep (
    department_id   BIGINT,
    department_name STRING,
    manager_id      BIGINT,
    location_id     BIGINT
) USING DELTA;

In [ ]:
%sql
-- Insert all records from the source DEPARTMENTS table into TRG_DEP
INSERT INTO workspace.hr.trg_dep (
    department_id,
    department_name,
    manager_id,
    location_id
)
SELECT
    departments.department_id,
    departments.department_name,
    departments.manager_id,
    departments.location_id
FROM
    workspace.hr.departments AS departments;

## Validation

In [ ]:
%sql
-- Count records in the target table
SELECT COUNT(*) AS total_records FROM workspace.hr.trg_dep;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** Original Oracle schema `HR` has been mapped to `workspace.hr`. Table names `TRG_DEP` and `DEPARTMENTS` have been lowercased to `trg_dep` and `departments` respectively.
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` Oracle hints have been removed as they are not applicable to Databricks Delta Lake and Spark SQL.
3.  **Table Creation:** A `CREATE TABLE IF NOT EXISTS` statement has been added for `workspace.hr.trg_dep`. The data types (`BIGINT`, `STRING`) were inferred based on typical Oracle HR schema definitions for `DEPARTMENTS` table columns (`NUMBER(4,0)`, `VARCHAR2(30)`, `NUMBER(6,0)`, `NUMBER(4,0)`).
4.  **Error Handling:** No explicit error handling (E$ tables) was present in the source SQL, so none was implemented here.
5.  **Parameters:** No ODI global or schema parameters (`#GLOBAL.`, `#SCHEMA.`) were found in the source SQL, so no Databricks widgets were created.
6.  **Optimization:** Consider adding `OPTIMIZE workspace.hr.trg_dep ZORDER BY (department_id);` if this table will be frequently queried with filters on `department_id`.
    *Note: If adding ZORDER, ensure to precede it with `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` in the same cell.*